# Notebook 9 — Visualization & Figures Module (Research Paper)

Generates all publication-quality figures as described in NEW_ONE.txt.

## Figure Sections
1. **Model Architecture Diagram** — Overall pipeline (Fig 1), Multi-task (Fig 2), Tier 0-4 pipeline
2. **Training Graphs** — Loss, Accuracy, F1, LR, SWA curves per dataset
3. **Performance Figures** — Confusion matrices, ROC, PR, Calibration, Reliability per dataset
4. **Explainability Figures** — Attention, SHAP, Lead importance, IG, GradCAM1D, SmoothGrad
5. **Cross-Dataset Figures** — PTB-XL → INCART comparison, generalization heatmap
6. **Embedding Analysis** — t-SNE, UMAP, PCA of shared embeddings
7. **Ablation Study Figures** — component contribution analysis
8. **Clinical Figures** — Risk distribution, rhythm, confidence, uncertainty

**Prerequisites:** All NB1–NB8 must have run and saved their output files.


In [ ]:
import torch, sys, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, f1_score, average_precision_score,
    roc_curve, precision_recall_curve, calibration_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")
FIGS_DIR = SAVE_DIR / "figures"
FIGS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(SAVE_DIR))
from ecg_model import ECGRiskNetXAI

with open(SAVE_DIR / "metadata.json") as f: META = json.load(f)
RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
LEAD_NAMES  = META.get("standard_lead_order",
    ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"])
CLS_COLORS  = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

# Load model
model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", map_location=device))
model.eval()

# Helper: safe JSON load
def safe_load(path):
    try:
        with open(path) as f: return json.load(f)
    except Exception:
        return {}

tr_results = safe_load(SAVE_DIR / "training_results.json")
cr_results = safe_load(SAVE_DIR / "crossdataset_results.json")
em_results = safe_load(SAVE_DIR / "edge_metrics.json")
ev_results = safe_load(SAVE_DIR / "train_eval_results.json")

print("✅ Setup complete.")
print(f"Figures will be saved to: {FIGS_DIR}")


## Figure 1: Architecture Diagram (Overall Pipeline)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 16))
ax.axis("off")

stages = [
    ("Input ECG (B, 12, 1000)",          "#ecf0f1", "black"),
    ("Conv Stem  →  (B, 64, 500)",       "#d5e8d4", "#27AE60"),
    ("InceptionTime Block\n[kernels 3/7/15/31]\n(B, 128, 500)", "#dae8fc", "#2980B9"),
    ("ResBlock ×2 + Downsample\n→ (B, 256, 250)",  "#dae8fc", "#2980B9"),
    ("SE Attention (lead-wise)\n→ (B, 256, 250)",  "#fff2cc", "#F39C12"),
    ("Transformer Encoder ×4\n→ (B, 125, 256)",    "#f8cecc", "#E74C3C"),
    ("Attention Pooling\n→ (B, 256)",              "#e1d5e7", "#8E44AD"),
    ("Metadata MLP (age, sex, HR)\n→ (B, 16)",     "#d5e8d4", "#27AE60"),
    ("Shared Embedding\n→ (B, 256)",               "#dae8fc", "#2980B9"),
    ("SupCon Projection  →  (B, 128)",             "#fff2cc", "#F39C12"),
    ("Multi-Task Heads:",                          "#f8cecc", "#E74C3C"),
    ("  Risk (4)  Arrhy (6)  MI (2)\n  ST/T (3)  Conduction (4)", "#f8cecc", "#E74C3C"),
    ("Loss: CBFocalLoss + 0.2 × SupConLoss",      "#ecf0f1", "black"),
    ("4-Tier Explainable Output",                  "#d5e8d4", "#27AE60"),
]

y_pos = 0.97
for text, fc, ec in stages:
    box = dict(boxstyle="round,pad=0.4", facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.text(0.5, y_pos, text, ha="center", va="top", fontsize=11,
            transform=ax.transAxes, bbox=box)
    if y_pos > 0.05:
        ax.annotate("", xy=(0.5, y_pos - 0.045),
                    xytext=(0.5, y_pos - 0.005),
                    xycoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    y_pos -= 0.066

ax.set_title("Figure 1 — ECGRiskNet-XAI Architecture (12-Lead Only)",
             fontsize=14, fontweight="bold", pad=10)
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig1_architecture.png", dpi=200, bbox_inches="tight")
plt.show()
print("✅ Figure 1 saved.")


## Figure 2: Multi-Task Learning Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis("off")

# Encoder box
enc_box = mpatches.FancyBboxPatch((0.1, 0.35), 0.25, 0.30,
    boxstyle="round,pad=0.02", facecolor="#dae8fc", edgecolor="#2980B9", linewidth=2)
ax.add_patch(enc_box)
ax.text(0.225, 0.50, "Shared\nEncoder\n(InceptionTime\n+Transformer)",
        ha="center", va="center", fontsize=11, fontweight="bold")

# Head boxes
heads = [
    ("Risk Head\n4 classes",        0.60, 0.80, "#d5e8d4", "#27AE60"),
    ("Arrhythmia Head\n6 classes",  0.60, 0.62, "#fff2cc", "#F39C12"),
    ("MI Head\n2 classes",          0.60, 0.44, "#f8cecc", "#E74C3C"),
    ("ST/T Head\n3 classes",        0.60, 0.26, "#e1d5e7", "#8E44AD"),
    ("Conduction Head\n4 classes",  0.60, 0.08, "#ffe6cc", "#D35400"),
]
for text, hx, hy, fc, ec in heads:
    b = mpatches.FancyBboxPatch((hx, hy), 0.22, 0.14,
        boxstyle="round,pad=0.02", facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(b)
    ax.text(hx + 0.11, hy + 0.07, text, ha="center", va="center", fontsize=10)
    ax.annotate("", xy=(hx, hy + 0.07),
                xytext=(0.35, 0.50),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

# SupCon projection
sc_box = mpatches.FancyBboxPatch((0.1, 0.02), 0.25, 0.12,
    boxstyle="round,pad=0.02", facecolor="#fff2cc", edgecolor="#F39C12", linewidth=2)
ax.add_patch(sc_box)
ax.text(0.225, 0.08, "SupCon\nProjection Head",
        ha="center", va="center", fontsize=10, fontweight="bold")
ax.annotate("", xy=(0.225, 0.14), xytext=(0.225, 0.35),
            arrowprops=dict(arrowstyle="->", color="orange", lw=1.5))

ax.set_title("Figure 2 — Multi-Task Learning Architecture with SupCon",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig2_multitask.png", dpi=200, bbox_inches="tight")
plt.show()
print("✅ Figure 2 saved.")


## Figure 3: Training Curves (PTB-XL)

In [ ]:
history = tr_results.get("history", {})
if history and "train_loss" in history:
    epochs = range(1, len(history["train_loss"]) + 1)
    best_ep = tr_results.get("best_epoch", len(epochs))

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    def plot_curve(ax, train_vals, val_vals, title, ylabel, best_ep=None):
        ax.plot(epochs, train_vals, label="Train", color="steelblue", linewidth=1.5)
        ax.plot(epochs, val_vals,   label="Val",   color="orange",    linewidth=1.5)
        if best_ep:
            ax.axvline(best_ep, color="red", linestyle="--", alpha=0.7, label=f"Best ep {best_ep}")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plot_curve(axes[0,0], history["train_loss"], history["val_loss"],
               "CB-Focal + SupCon Loss", "Loss", best_ep)
    plot_curve(axes[0,1], history["train_acc"],  history["val_acc"],
               "Accuracy", "Accuracy", best_ep)
    plot_curve(axes[0,2], history["train_f1"],   history["val_f1"],
               "Macro F1 Score", "Macro F1", best_ep)

    axes[1,0].plot(epochs, history["lr"], color="green", linewidth=1.5)
    axes[1,0].set_title("Learning Rate Schedule", fontweight="bold")
    axes[1,0].set_xlabel("Epoch"); axes[1,0].set_ylabel("LR")
    axes[1,0].grid(True, alpha=0.3)

    # Val F1 zoomed
    axes[1,1].plot(epochs, history["val_f1"], color="orange", linewidth=1.5)
    axes[1,1].axvline(best_ep, color="red", linestyle="--", label=f"Best: {max(history['val_f1']):.4f}")
    axes[1,1].set_title("Val Macro F1 (zoomed)", fontweight="bold")
    axes[1,1].set_xlabel("Epoch"); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

    # Loss gap (train-val)
    loss_gap = [abs(a - b) for a, b in zip(history["train_loss"], history["val_loss"])]
    axes[1,2].plot(epochs, loss_gap, color="purple", linewidth=1.5)
    axes[1,2].set_title("Loss Gap |Train - Val|", fontweight="bold")
    axes[1,2].set_xlabel("Epoch"); axes[1,2].set_ylabel("|Loss gap|"); axes[1,2].grid(True, alpha=0.3)

    plt.suptitle("Figure 3 — PTB-XL Training History — ECGRiskNet-XAI",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGS_DIR / "fig3_training_curves.png", dpi=150)
    plt.show()
    print("✅ Figure 3 saved.")
else:
    print("⚠️  Training history not found — run NB4 first.")


## Figure 4: Confusion Matrices — PTB-XL Train / Val / Test + INCART

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def get_preds_probs(npz_path):
    """Run model on split and return (preds, probs, true)."""
    d = np.load(npz_path)
    X = d["X"].astype(np.float32)
    y = d["y"].astype(np.int64)
    M = d["meta"].astype(np.float32) if "meta" in d else np.zeros((len(y), 3), np.float32)
    X_t = torch.from_numpy(X.transpose(0, 2, 1))
    M_t = torch.from_numpy(M)
    loader = DataLoader(TensorDataset(X_t, M_t), batch_size=64, num_workers=0)
    preds, probs = [], []
    model.eval()
    with torch.no_grad():
        for xb, mb in loader:
            out = model(xb.to(device), mb.to(device))
            p   = torch.softmax(out["risk"], 1).cpu().numpy()
            probs.append(p)
            preds.extend(p.argmax(axis=1))
    return np.array(preds), np.vstack(probs), y


splits_data = []
for split, fname in [("Train","train_processed.npz"),("Val","val_processed.npz"),("Test","test_processed.npz")]:
    fp = SAVE_DIR / fname
    if fp.exists():
        p, pr, y = get_preds_probs(fp)
        splits_data.append((split, p, pr, y))
    else:
        print(f"⚠️  {fname} not found")

cmaps = ["Blues", "Oranges", "Greens"]
n_plots = len(splits_data)
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))
if n_plots == 1: axes = [axes]

for ax, (name, preds, probs, true), cmap in zip(axes, splits_data, cmaps):
    cm = confusion_matrix(true, preds, labels=[0,1,2,3])
    acc = (preds == true).mean()
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["Low","Moderate","High","Critical"],
                yticklabels=["Low","Moderate","High","Critical"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"PTB-XL {name}  (Acc={acc*100:.1f}%)", fontweight="bold")

plt.suptitle("Figure 4 — Confusion Matrices per Split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig4_confusion_matrices.png", dpi=150)
plt.show()
print("✅ Figure 4 saved.")


## Figure 5: ROC and PR Curves per Split

In [ ]:
fig, axes = plt.subplots(len(splits_data), 2, figsize=(14, 5 * len(splits_data)))
if len(splits_data) == 1: axes = [axes]

for row, (name, preds, probs, true) in enumerate(splits_data):
    try:
        y_bin = label_binarize(true, classes=[0,1,2,3])
        for k, color in zip(range(4), CLS_COLORS):
            fpr, tpr, _ = roc_curve(y_bin[:, k], probs[:, k])
            auc_k = roc_auc_score(y_bin[:, k], probs[:, k])
            axes[row][0].plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} ({auc_k:.2f})")
        axes[row][0].plot([0,1],[0,1],"k--",alpha=0.3)
        axes[row][0].set_title(f"ROC — PTB-XL {name}", fontweight="bold")
        axes[row][0].set_xlabel("FPR"); axes[row][0].set_ylabel("TPR")
        axes[row][0].legend(fontsize=8); axes[row][0].grid(True, alpha=0.3)

        for k, color in zip(range(4), CLS_COLORS):
            prec_c, rec_c, _ = precision_recall_curve(y_bin[:, k], probs[:, k])
            ap_k = average_precision_score(y_bin[:, k], probs[:, k])
            axes[row][1].plot(rec_c, prec_c, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.2f})")
        axes[row][1].set_title(f"PR — PTB-XL {name}", fontweight="bold")
        axes[row][1].set_xlabel("Recall"); axes[row][1].set_ylabel("Precision")
        axes[row][1].legend(fontsize=8); axes[row][1].grid(True, alpha=0.3)
    except Exception as e:
        print(f"  {name}: {e}")

plt.suptitle("Figure 5 — ROC and PR Curves per PTB-XL Split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig5_roc_pr_curves.png", dpi=150)
plt.show()
print("✅ Figure 5 saved.")


## Figure 6: Cross-Dataset Performance Comparison

In [ ]:
dataset_names = ["PTB-XL\nTrain", "PTB-XL\nVal", "PTB-XL\nTest", "INCART"]

def get_metric(results, key, default=0.0):
    return float(results.get(key, default))

te = ev_results.get("ptbxl_test",  {})
tr = ev_results.get("ptbxl_train", {})
vl = ev_results.get("ptbxl_val",   {})

accs  = [get_metric(tr,"accuracy"),   get_metric(vl,"accuracy"),
          get_metric(te,"accuracy"),   get_metric(cr_results,"incart_accuracy")]
f1s   = [get_metric(tr,"macro_f1"),   get_metric(vl,"macro_f1"),
          get_metric(te,"macro_f1"),   get_metric(cr_results,"incart_macro_f1")]
aucs  = [get_metric(tr,"roc_auc"),    get_metric(vl,"roc_auc"),
          get_metric(te,"roc_auc"),    get_metric(cr_results,"incart_roc_auc")]

x = np.arange(len(dataset_names))
width = 0.25
bar_colors = ["steelblue", "orange", "green", "purple"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, vals, ylabel, title in [
    (axes[0], [a*100 for a in accs], "Accuracy (%)", "Accuracy"),
    (axes[1], f1s,  "Macro F1",  "Macro F1 Score"),
    (axes[2], aucs, "ROC-AUC",   "ROC-AUC"),
]:
    bars = ax.bar(dataset_names, vals, color=bar_colors, edgecolor="white", alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003 * (100 if "Acc" in ylabel else 1),
                f"{val:.2f}" if "Acc" not in ylabel else f"{val:.1f}%",
                ha="center", fontsize=9, fontweight="bold")
    ax.set_title(f"{title} — All Datasets", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(0, max(vals) * 1.18 if vals else 1.0)

plt.suptitle("Figure 6 — Cross-Dataset Performance Comparison",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig6_cross_dataset_comparison.png", dpi=150)
plt.show()
print("✅ Figure 6 saved.")


## Figure 7: Embedding Analysis — t-SNE & PCA of Shared Embeddings

In [ ]:
# Extract embeddings from test set
test_d = np.load(SAVE_DIR / "test_processed.npz")
X_test = test_d["X"].astype(np.float32)
y_test = test_d["y"].astype(np.int64)
M_test = test_d["meta"].astype(np.float32) if "meta" in test_d \
         else np.zeros((len(y_test), 3), np.float32)

n_embed = min(1000, len(X_test))   # use at most 1000 samples for speed
idx_e   = np.random.choice(len(X_test), n_embed, replace=False)

X_e = torch.from_numpy(X_test[idx_e].transpose(0, 2, 1)).to(device)
M_e = torch.from_numpy(M_test[idx_e]).to(device)
y_e = y_test[idx_e]

embeddings = []
model.eval()
with torch.no_grad():
    loader_e = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_e.cpu(), M_e.cpu()),
        batch_size=64
    )
    for xb, mb in loader_e:
        out = model(xb.to(device), mb.to(device))
        embeddings.append(out["embedding"].cpu().numpy())

embeddings = np.vstack(embeddings)   # (n_embed, 256)
print(f"Embeddings shape: {embeddings.shape}")

# PCA
pca = PCA(n_components=2, random_state=SEED)
emb_pca = pca.fit_transform(embeddings)

# t-SNE
print("Running t-SNE (may take 1–2 min)...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_tsne = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
markers = ["o", "s", "^", "D"]

for ax, emb, title in [(axes[0], emb_pca, "PCA"), (axes[1], emb_tsne, "t-SNE")]:
    for k in range(4):
        mask = y_e == k
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=CLS_COLORS[k], marker=markers[k],
                   label=RISK_LABELS[k], alpha=0.6, s=25, edgecolors="none")
    ax.set_title(f"{title} — Shared Embeddings", fontweight="bold")
    ax.legend(fontsize=10); ax.grid(True, alpha=0.2)
    ax.set_xlabel(f"{title} 1"); ax.set_ylabel(f"{title} 2")

plt.suptitle("Figure 7 — Embedding Analysis (PCA & t-SNE)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig7_embeddings.png", dpi=150)
plt.show()
print("✅ Figure 7 saved.")


## Figure 8: Ablation Study Summary

In [ ]:
# Ablation study table (use simulated values if ablation experiments not run)
# Replace these with actual measured values when ablation experiments are complete.
ablation_components = [
    "Full Model\n(Ours)",
    "w/o Transformer\n(TCN only)",
    "w/o InceptionTime\n(Simple CNN)",
    "w/o SE Attention",
    "w/o SupCon Loss",
    "w/o Metadata\nFusion",
    "w/o Multi-Task\nHeads",
    "FocalLoss\n(not CB-Focal)",
]

# These are illustrative placeholder values.
# Replace with real measured values from separate ablation training runs.
ablation_acc = [0.0]*8   # placeholder
ablation_f1  = [
    tr_results.get("test_macro_f1", 0.85),   # full model
    tr_results.get("test_macro_f1", 0.85) * 0.93,
    tr_results.get("test_macro_f1", 0.85) * 0.90,
    tr_results.get("test_macro_f1", 0.85) * 0.96,
    tr_results.get("test_macro_f1", 0.85) * 0.95,
    tr_results.get("test_macro_f1", 0.85) * 0.97,
    tr_results.get("test_macro_f1", 0.85) * 0.94,
    tr_results.get("test_macro_f1", 0.85) * 0.92,
]

ablation_colors = ["#2ecc71"] + ["#e74c3c"] * (len(ablation_components) - 1)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(ablation_components, ablation_f1,
              color=ablation_colors, edgecolor="white", alpha=0.85)
for bar, val in zip(bars, ablation_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")

ax.axhline(ablation_f1[0], color="green", linestyle="--", alpha=0.5,
           label=f"Full Model: {ablation_f1[0]:.3f}")
ax.set_title("Figure 8 — Ablation Study: Macro F1 Contribution per Component",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Macro F1 Score")
ax.set_ylim(0, min(1.0, max(ablation_f1) * 1.15))
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis="y")

note_ax = ax.twinx()
note_ax.set_yticks([])
note_ax.set_ylabel("* Values are estimated placeholders — replace with real ablation runs",
                   fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(FIGS_DIR / "fig8_ablation.png", dpi=150)
plt.show()
print("✅ Figure 8 saved.")


## Figure 9: Clinical Distribution Figures

In [ ]:
# Load test set for distribution analysis
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Risk distribution — test set
cnt_test = Counter(y_test)
axes[0,0].pie(
    [cnt_test.get(k,0) for k in range(4)],
    labels=[RISK_LABELS[k] for k in range(4)],
    colors=CLS_COLORS, autopct="%1.1f%%", startangle=140,
    textprops={"fontsize": 11}
)
axes[0,0].set_title("Risk Distribution — Test Set", fontweight="bold")

# Confidence distribution (requires batch predictions)
preds_test, probs_test, _ = get_preds_probs(SAVE_DIR / "test_processed.npz")
confidences = probs_test.max(axis=1)

for k, color in enumerate(CLS_COLORS):
    mask = preds_test == k
    if mask.sum() > 0:
        axes[0,1].hist(confidences[mask], bins=20, alpha=0.5,
                       color=color, label=RISK_LABELS[k])
axes[0,1].set_title("Confidence Distribution per Predicted Class", fontweight="bold")
axes[0,1].set_xlabel("Confidence"); axes[0,1].set_ylabel("Count")
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# Error distribution (wrong predictions)
errors = preds_test != y_test
if errors.sum() > 0:
    err_conf = confidences[errors]
    axes[0,2].hist(err_conf, bins=20, color="red", alpha=0.7, edgecolor="white")
    axes[0,2].set_title("Confidence of Wrong Predictions", fontweight="bold")
    axes[0,2].set_xlabel("Confidence"); axes[0,2].set_ylabel("Count")
    axes[0,2].grid(True, alpha=0.3)

# Per-class accuracy
per_cls_acc = [(preds_test[y_test==k] == k).mean() if (y_test==k).sum() > 0 else 0
               for k in range(4)]
axes[1,0].bar([RISK_LABELS[k] for k in range(4)], per_cls_acc,
              color=CLS_COLORS, edgecolor="white")
axes[1,0].set_title("Per-Class Accuracy", fontweight="bold")
axes[1,0].set_ylabel("Accuracy"); axes[1,0].set_ylim(0, 1.15)
axes[1,0].grid(True, alpha=0.3, axis="y")
for i, (bar, val) in enumerate(zip(axes[1,0].patches, per_cls_acc)):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, val+0.01, f"{val:.2f}",
                   ha="center", fontsize=10)

# Calibration — test set
try:
    y_bin = label_binarize(y_test, classes=[0,1,2,3])
    for k, color in enumerate(CLS_COLORS):
        frac, mean_pred = calibration_curve(
            y_bin[:, k], probs_test[:, k], n_bins=10, strategy="uniform"
        )
        axes[1,1].plot(mean_pred, frac, "o-", color=color, label=RISK_LABELS[k])
    axes[1,1].plot([0,1],[0,1],"k--",alpha=0.4,label="Perfect")
    axes[1,1].set_title("Calibration Curve — Test Set", fontweight="bold")
    axes[1,1].set_xlabel("Mean Predicted Prob"); axes[1,1].set_ylabel("Fraction of Positives")
    axes[1,1].legend(fontsize=9); axes[1,1].grid(True, alpha=0.3)
except Exception as e:
    print(f"Calibration skipped: {e}")

# Support distribution
support = [int((y_test == k).sum()) for k in range(4)]
axes[1,2].bar([RISK_LABELS[k] for k in range(4)], support,
              color=CLS_COLORS, edgecolor="white")
axes[1,2].set_title("Class Support — Test Set", fontweight="bold")
axes[1,2].set_ylabel("Count"); axes[1,2].grid(True, alpha=0.3, axis="y")
for bar, val in zip(axes[1,2].patches, support):
    axes[1,2].text(bar.get_x()+bar.get_width()/2, val+5, str(val),
                   ha="center", fontsize=10)

plt.suptitle("Figure 9 — Clinical Distribution Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig9_clinical_distribution.png", dpi=150)
plt.show()
print("✅ Figure 9 saved.")


## Figure 10: Edge-AI Optimization Summary

In [ ]:
if em_results:
    methods = list(em_results.keys())
    lats    = [em_results[m].get("latency_ms", 0) for m in methods]
    sizes   = [em_results[m].get("size_mb",    0) for m in methods]
    f1s_e   = [em_results[m].get("macro_f1",   0) for m in methods]
    speedups = [lats[0] / max(l, 0.001) for l in lats]

    ec      = ["steelblue","orange","green","purple"]
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    for ax, vals, ylabel, title in [
        (axes[0], lats,     "ms/sample",  "Inference Latency"),
        (axes[1], sizes,    "MB",          "Model Size"),
        (axes[2], speedups, "×",           "Speedup vs Baseline"),
        (axes[3], f1s_e,    "Macro F1",    "Accuracy Preservation"),
    ]:
        bars = ax.bar(methods, vals, color=ec[:len(methods)], edgecolor="white", alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                    f"{val:.2f}", ha="center", fontsize=9)
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)

    plt.suptitle("Figure 10 — Edge-AI Optimization Results", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGS_DIR / "fig10_edge_ai.png", dpi=150)
    plt.show()
    print("✅ Figure 10 saved.")
else:
    print("⚠️  Edge metrics not found — run NB7 first.")


## Figure Index & Summary

In [ ]:
print("NOTEBOOK 9 COMPLETE ✅")
print(f"\nAll figures saved to: {FIGS_DIR}")
print()
print("Figure Index:")
figure_list = [
    ("fig1_architecture.png",            "Overall Hybrid Architecture Pipeline"),
    ("fig2_multitask.png",               "Multi-Task Learning + SupCon Diagram"),
    ("fig3_training_curves.png",         "PTB-XL Training History (6 panels)"),
    ("fig4_confusion_matrices.png",      "Confusion Matrices — Train / Val / Test"),
    ("fig5_roc_pr_curves.png",           "ROC & PR Curves per Split"),
    ("fig6_cross_dataset_comparison.png","Cross-Dataset Acc/F1/AUC Bar Chart"),
    ("fig7_embeddings.png",              "PCA & t-SNE of Shared Embeddings"),
    ("fig8_ablation.png",               "Ablation Study — Component F1 Contribution"),
    ("fig9_clinical_distribution.png",   "Clinical Distribution + Calibration"),
    ("fig10_edge_ai.png",               "Edge-AI Optimization Results"),
]
for fname, desc in figure_list:
    fp = FIGS_DIR / fname
    status = "✅" if fp.exists() else "⚠️  (not generated)"
    print(f"  {status}  {fname:<45}  {desc}")

print()
print("Additional figures from other notebooks:")
additional = [
    ("explainability/attention_maps_all_blocks.png", "Attention Maps (NB5)"),
    ("explainability/gradcam1d.png",                "GradCAM1D Heatmaps (NB5)"),
    ("explainability/smoothgrad.png",               "SmoothGrad Saliency (NB5)"),
    ("explainability/integrated_gradients.png",     "Integrated Gradients (NB5)"),
    ("explainability/lead_importance.png",          "Lead Importance Bar (NB5)"),
    ("incart_evaluation.png",                       "INCART Eval (NB6)"),
    ("4tier_result_class0.png",                     "4-Tier Report — Low (NB8)"),
    ("4tier_result_class3.png",                     "4-Tier Report — Critical (NB8)"),
]
for fname, desc in additional:
    fp = SAVE_DIR / fname
    status = "✅" if fp.exists() else "⚠️  (not generated)"
    print(f"  {status}  {fname:<50}  {desc}")

print()
print("=" * 60)
print("ALL 9 NOTEBOOKS COMPLETE ✅")
print("ECGRiskNet-XAI — 4-Tier Explainable AI for 12-Lead ECG")
print("=" * 60)
